# 02 · Trích xuất + chấm điểm (đúng code ai-service)

Mỗi lượt = một model × một ngôn ngữ: trích xuất CV (LLM #1) → Python tính số năm, lọc hồ sơ → chấm (LLM #2) → Python tính điểm.
Model: `google/gemini-3.5-flash-lite` (sản phẩm) và `openai/gpt-5.4-nano-2026-03-17` (so sánh).

**Input cần đính kèm:** dataset `ats-cv-scoring-benchmark` + output của notebook `01_translate` (bản dịch tiếng Việt).

**Chạy tiếp được:** hết quota giữa chừng thì chạy lại (đính kèm thêm output của lần chạy trước); cặp đã chấm ok được bỏ qua.

In [ ]:
# Tìm dataset đã đính kèm (code ai-service + dữ liệu) trong /kaggle/input
import importlib, json, subprocess, sys
from pathlib import Path

INPUT, WORK = Path("/kaggle/input"), Path("/kaggle/working")
engine_file = next(INPUT.rglob("ats_code/app/services/scoring/engine.py"), None)
assert engine_file, "Chưa đính kèm dataset ats-cv-scoring-benchmark (Add Input → Datasets)"
CODE = engine_file.parents[3]                     # .../ats_code
DATA = CODE.parent / "data"
sys.path.insert(0, str(CODE))

if importlib.util.find_spec("pydantic_settings") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pydantic-settings"], check=True)

import kaggle_benchmarks as kbench
from benchmark.common import read_jsonl
from benchmark import kaggle_runner as kr

pairs = read_jsonl(DATA / "pairs.jsonl")
texts_en = {t["hash"]: t for t in read_jsonl(DATA / "texts_en.jsonl")}
print("kaggle_benchmarks", getattr(kbench, "__version__", "?"), "| code:", CODE)
print(len(pairs), "cặp |", len(texts_en), "văn bản duy nhất")

## Cấu hình
- `SMOKE_LIMIT = 2`: chạy thử 2 cặp mỗi lượt để kiểm tra trước; `None` để chạy toàn bộ.
- `RUNS`: các lượt cần chạy — có thể tách Gemini / GPT ra hai lần chạy chạy song song.
- `STABILITY`: chấm lặp thêm 2 lần 50 cặp test (Gemini, tiếng Việt) để đo độ ổn định.

In [ ]:
SMOKE_LIMIT = None
RUNS = [("gemini", "en"), ("gemini", "vi"), ("gpt", "en"), ("gpt", "vi")]
STABILITY = True

In [ ]:
vi = kr.load_translations([INPUT, WORK])
texts = {"en": {h: t["text"] for h, t in texts_en.items()}, "vi": vi}
print("Bản dịch tiếng Việt:", len(vi), "/", len(texts_en))

all_stats = {}
for model_key, lang in RUNS:
    cfg = kr.RunConfig(model_key=model_key, lang=lang, out_dir=WORK, prior_roots=[INPUT])
    all_stats[cfg.score_file] = kr.run_scoring(cfg, pairs, texts[lang], limit=SMOKE_LIMIT)
    print(cfg.score_file, all_stats[cfg.score_file])

In [ ]:
if STABILITY:
    subset = [p for p in pairs if p["split"] == "test"][:50]
    for tag in ("r2", "r3"):
        cfg = kr.RunConfig(model_key="gemini", lang="vi", out_dir=WORK, prior_roots=[INPUT], tag=tag)
        all_stats[cfg.score_file] = kr.run_scoring(cfg, subset, texts["vi"], limit=SMOKE_LIMIT)
        print(cfg.score_file, all_stats[cfg.score_file])

## Tổng kết (tính cả kết quả của các lần chạy trước)

In [ ]:
from benchmark.common import find_files
for name in sorted({p.name for p in find_files([INPUT, WORK], "scores.*.jsonl")}):
    rows = [r for p in find_files([INPUT, WORK], name) for r in read_jsonl(p)]
    ok = {r["pair_id"] for r in rows if r.get("ok")}
    print(f"{name:32} ok={len(ok)}  lỗi (chưa có kết quả ok)={len({r['pair_id'] for r in rows} - ok)}")